# Assignment 2B — RAG Pipeline
Domain: NIST cybersecurity guidance (SP 800-37 Rev. 2, SP 800-30 Rev. 1, SP 800-61 Rev. 2), reused from Assignment 1.

**Reused from Assignment 1:** the cleaned domain `.txt` files (document collection), 10 queries with their expected answers
(from the Assignment 1 held-out set, topped up from the training pairs if fewer than 10), and the fine-tuned Qwen2.5-1.5B-Instruct
adapter as the answer generator. No new data was collected.

**How to read this notebook:** every table below is computed by the code that precedes it. The short written inferences are
generated from those measured numbers by explicit rules, so they always match the tables. Two cells (D2 spot-check) ask you to
type your own verdicts after reading the evidence.

## 0. Setup

In [ ]:
# ---- Config (set before importing transformers so downloads go to the data volume) --------------
import os, sys, json, re, time, random, math, shutil, subprocess, platform
from pathlib import Path
from collections import Counter

PROJ = Path(os.environ.get("A1B_PROJ", Path.home() / "grp-165-datavol-1" / "a1b")).resolve()
os.environ.setdefault("HF_HOME", str(PROJ / "cache" / "hf"))
CORPUS_DIR = Path(os.environ.get("A1B_CORPUS", PROJ.parent / "submission" / "domain_text"))   # cleaned .txt files from Assignment 1
STAGE1, STAGE2 = PROJ / "stage1", PROJ / "stage2"
OUT = PROJ / "stage_2b"; OUT.mkdir(parents=True, exist_ok=True)

EMBED_MODEL  = os.environ.get("A1B_EMBED",  "BAAI/bge-small-en-v1.5")                  # 512-token limit (see Part A)
RERANK_MODEL = os.environ.get("A1B_RERANK", "cross-encoder/ms-marco-MiniLM-L-6-v2")
GEN_BASE     = os.environ.get("A1B_MODEL",  "Qwen/Qwen2.5-1.5B-Instruct")              # Assignment 1 base model
ADAPTER_DIR  = Path(os.environ.get("A1B_ADAPTER", STAGE2 / "lora_adapter"))             # Assignment 1 fine-tuned adapter
FALLBACK_GEN = os.environ.get("A1B_FALLBACK", "Qwen/Qwen2.5-3B-Instruct")              # used only if the Assignment 1 model fails D1
FALLBACK_MODE = os.environ.get("A1B_FALLBACK_MODE", "auto")                             # auto | never | always
JUDGE_ENV    = os.environ.get("A1B_JUDGE", "")                                          # empty = pick from free disk space

SEED, N_QUERIES = 42, 10
CHUNK_SIZES = [100, 400]       # tokens, no overlap (Part A2)
BATCH_SIZE  = 32               # embedding batch size
TOP_K, FUSE_N, RRF_K = 5, 20, 60
RERANK_N, FINAL_K = 10, 3
MAX_NEW = 200                  # maximum new tokens for an answer (the judge reply is capped at 150)
QUERY_PREFIX = ("Represent this sentence for searching relevant passages: " if "bge" in EMBED_MODEL.lower() else "")

print("Project folder :", PROJ)
print("Corpus folder  :", CORPUS_DIR, "(exists)" if CORPUS_DIR.exists() else "(missing - will rebuild from stage1/chunks.jsonl)")
print("Output folder  :", OUT)

In [ ]:
# ---- Install the three extra libraries into THIS kernel's environment, keeping the pinned stack unchanged ----
import importlib.util, importlib.metadata as im
NEED = {"sentence_transformers": "sentence-transformers==3.3.1", "faiss": "faiss-cpu==1.9.0.post1", "rank_bm25": "rank-bm25==0.2.2"}
missing = [spec for mod, spec in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    pins = []
    for pkg in ["torch", "transformers", "tokenizers", "peft", "accelerate", "bitsandbytes", "trl", "huggingface-hub", "safetensors", "numpy"]:
        try: pins.append(f"{pkg}=={im.version(pkg)}")
        except im.PackageNotFoundError: pass
    cfile = OUT / "constraints.txt"; cfile.write_text("\n".join(pins) + "\n")
    print("Installing:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(cfile), *missing], capture_output=True, text=True)
    print(r.stdout[-800:]); 
    if r.returncode != 0: print(r.stderr[-1500:]); raise SystemExit("pip install failed - see message above")
    importlib.invalidate_caches()
else:
    print("sentence-transformers, faiss and rank_bm25 already installed.")

In [ ]:
# ---- Environment check -----------------------------------------------------------------------------
import numpy as np, torch, transformers, faiss
import sentence_transformers, rank_bm25
from IPython.display import Markdown, display

device = "cuda" if torch.cuda.is_available() else "cpu"
def gpu_snapshot():
    try:
        o = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=10).stdout.strip().split(",")
        return {"mem_used_mib": int(o[0]), "mem_total_mib": int(o[1]), "util_pct": int(o[2])}
    except Exception: return None
print({"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "sentence-transformers": sentence_transformers.__version__, "faiss": faiss.__version__, "device": device})
if device == "cuda": print("GPU:", torch.cuda.get_device_name(0), "| other users on the GPU right now:", gpu_snapshot())
free_gb = shutil.disk_usage(PROJ).free / 1e9
print(f"Free disk on the data volume: {free_gb:.1f} GB")
if free_gb < 8: print("WARNING: low disk space. Delete a1b/stage3/merged_bf16, a1b/stage2/checkpoints and unused folders under a1b/cache/hf/hub before continuing.")

def read_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def norm(s): return re.sub(r"\s+", " ", s).strip()
def contains(chunk_text, span): return norm(span) in norm(chunk_text)
def md(s): display(Markdown(s))
def esc(s): return str(s).replace("|", "\\|").replace("\n", " ")
def md_table(headers, rows):
    return "\n".join(["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)] + ["| " + " | ".join(esc(c) for c in r) + " |" for r in rows])

### 0.1 Document collection (cleaned `.txt` files from Assignment 1)

In [ ]:
def load_corpus():
    docs = {}
    files = sorted(CORPUS_DIR.glob("*.txt")) if CORPUS_DIR.exists() else []
    if files:
        for p in files:
            raw = re.sub(r"(?m)^\[pages \d+-\d+\]\s*$", "", p.read_text(encoding="utf-8"))   # drop the page markers added at export
            docs[p.stem] = norm(raw)
        return docs, f"{len(files)} files in {CORPUS_DIR}"
    chunks = [c for c in read_jsonl(STAGE1 / "chunks.jsonl") if c.get("page_end", 99) > 3]   # same rule as the export of the .txt files
    by = {}
    for c in sorted(chunks, key=lambda c: (c["doc_id"], c["page_start"])): by.setdefault(c["doc_id"], []).append(c["text"])
    return {d: norm(" ".join(t)) for d, t in by.items()}, "rebuilt from stage1/chunks.jsonl"
DOCS, src = load_corpus()
assert DOCS, "no corpus found - check A1B_CORPUS / A1B_PROJ"
print("Corpus source:", src)
md(md_table(["document", "characters", "words"], [[d, f"{len(t):,}", f"{len(t.split()):,}"] for d, t in DOCS.items()]))

### 0.2 The 10 queries and expected answers (from the Assignment 1 evaluation set)

In [ ]:
meta = {norm(r["instruction"]): r for r in read_jsonl(STAGE1 / "domain_qa_with_meta.jsonl")}
src_chunks = {c["chunk_id"]: c for c in read_jsonl(STAGE1 / "chunks.jsonl")}
val, train = read_jsonl(STAGE2 / "val.jsonl"), read_jsonl(STAGE2 / "train.jsonl")
rng = random.Random(SEED)
pool_val = val[:]; rng.shuffle(pool_val)
chosen = [("held-out", r) for r in pool_val[:N_QUERIES]]
if len(chosen) < N_QUERIES:      # the Assignment 1 held-out set has fewer than 10 pairs: top up from the training pairs, balanced over documents
    pool = train[:]; rng.shuffle(pool); counts = Counter(r["doc_id"] for _, r in chosen)
    while len(chosen) < N_QUERIES:
        pool.sort(key=lambda r: counts[r["doc_id"]])        # stable: keeps the shuffled order among ties
        r = pool.pop(0); counts[r["doc_id"]] += 1; chosen.append(("training", r))
QUERIES = []
for i, (srcname, r) in enumerate(chosen, 1):
    m = meta[norm(r["instruction"])]
    QUERIES.append({"n": i, "source": srcname, "doc_id": m["doc_id"], "type": m["type"], "question": r["instruction"],
                    "expected": r["response"], "chunk_id": m["chunk_id"], "pages": m.get("pages"),
                    "source_text": norm(src_chunks[m["chunk_id"]]["text"])})
n_h = sum(q["source"] == "held-out" for q in QUERIES)
md(f"**{n_h} held-out + {len(QUERIES) - n_h} training-set questions** from Assignment 1 "
   f"(the Assignment 1 held-out set has {len(val)} pairs, fewer than the 10 needed).\n\n" +
   md_table(["#", "source", "document", "type", "question", "expected answer"],
            [[q["n"], q["source"], q["doc_id"], q["type"], q["question"], q["expected"]] for q in QUERIES]))

### 0.3 Answer span for each query
For each query, the **answer span** is the sentence (or two) in the corpus that answers it. A retrieved chunk is a **hit** if it contains that span.
The span is proposed automatically from the passage the question was written from (best word overlap with the expected answer),
saved to `a1b/stage_2b/answer_spans.json`, and checked below to be present word-for-word in the corpus. If a span looks wrong,
edit it in that file (keep the text exactly as in the corpus) and re-run from this cell.

In [ ]:
STOP = set("a an the of to and or in on for with by is are was were be been being as at from that this these those it its their there "
           "which who whom what when where how why can may must should shall will would not no".split())
def content(s): return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 2}
def sentences(t): return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9(\[\"'])", t) if s.strip()]
def propose_span(q):
    sents = sentences(q["source_text"]); a, qq = content(q["expected"]), content(q["question"]); best = None
    for i in range(len(sents)):
        for j in (i, i + 1):
            if j >= len(sents): continue
            cand = " ".join(sents[i:j + 1]); n = len(cand.split())
            if n > 80 or n < 5: continue
            c = content(cand)
            score = len(a & c) / max(1, len(a)) + 0.3 * len(qq & c) / max(1, len(qq)) - (0.15 if j > i else 0) - 0.002 * n
            if best is None or score > best[0]: best = (score, cand)
    return best[1] if best else (sents[0] if sents else q["source_text"][:300])

SPAN_FILE = OUT / "answer_spans.json"
saved = json.loads(SPAN_FILE.read_text()) if SPAN_FILE.exists() else {}
reused = bool(saved) and all(saved.get(str(q["n"]), {}).get("question") == q["question"] for q in QUERIES)
if not reused: saved = {str(q["n"]): {"question": q["question"], "span": propose_span(q)} for q in QUERIES}; SPAN_FILE.write_text(json.dumps(saved, indent=1))
for q in QUERIES:
    q["span"] = norm(saved[str(q["n"])]["span"])
    assert q["span"] in DOCS[q["doc_id"]], f"Query {q['n']}: the span is not in the corpus text - fix {SPAN_FILE}"
print("Spans", "loaded from" if reused else "proposed and saved to", SPAN_FILE)
md(md_table(["#", "question", "answer span (from " + "the corpus)"], [[q["n"], q["question"], q["span"]] for q in QUERIES]))

## Part A — Dense search and chunk size

In [ ]:
# ---- Embedding model, its limit, and token counting with its own tokenizer -------------------------
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL, device=device)
tokenizer = embedder.tokenizer; tokenizer.model_max_length = 10 ** 9        # silence the long-text warning; chunks are cut by us
assert tokenizer.is_fast, "a fast tokenizer is needed for character offsets"
MAX_LEN = embedder.max_seq_length
print(f"Embedding model: {EMBED_MODEL} | maximum input length: {MAX_LEN} tokens | vector size: {embedder.get_sentence_embedding_dimension()}")
md("**Why this embedding model:** the brief's example, all-MiniLM-L6-v2, accepts only 256 tokens, so a 400-token chunk would be cut off when embedded. "
   f"I therefore use `{EMBED_MODEL}`, whose limit is {MAX_LEN} tokens, so both 100- and 400-token chunks fit (the two special tokens add 2). "
   "No chunk size had to be replaced.")
for s in CHUNK_SIZES: assert s + 2 <= MAX_LEN, f"{s} tokens exceeds the model limit {MAX_LEN} - switch model or replace this size"

### A1. Build dense search

In [ ]:
def chunk_fixed(docs, size, tok):
    """Fixed-size chunks of `size` tokens (tokenizer of the embedding model), no overlap, cut from the original text by character offsets."""
    chunks = []
    for doc_id, text in docs.items():
        offs = tok(text, add_special_tokens=False, return_offsets_mapping=True, truncation=False)["offset_mapping"]
        for s in range(0, len(offs), size):
            e = min(s + size, len(offs)); t = text[offs[s][0]:offs[e - 1][1]].strip()
            if t: chunks.append({"id": len(chunks), "doc_id": doc_id, "text": t, "n_tokens": e - s})
    return chunks

def build_dense_search(chunks, model, batch_size=BATCH_SIZE, query_prefix=QUERY_PREFIX):
    """Embed the chunks, build a FAISS IndexFlatIP and return (search, info); search(query, k) -> [(chunk, score)] top-k."""
    model.encode(["warm-up"], normalize_embeddings=True)                      # exclude one-off start-up cost from the timing
    t0 = time.perf_counter()
    emb = model.encode([c["text"] for c in chunks], batch_size=batch_size, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)                    # unit vectors, so inner product = cosine similarity
    info = {"build_seconds": time.perf_counter() - t0, "dim": int(emb.shape[1]), "n_chunks": len(chunks), "batch_size": batch_size}
    def search(query, k=TOP_K):
        q = model.encode([query_prefix + query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).astype("float32")
        scores, ids = index.search(q, min(k, len(chunks)))
        return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]
    return search, info
print("Defined chunk_fixed() and build_dense_search().")

### A2. Choose a chunk size (fixed-size chunks of 100 and 400 tokens, no overlap)

In [ ]:
from transformers import AutoTokenizer, AutoConfig
qtok = AutoTokenizer.from_pretrained(GEN_BASE)                                  # Assignment 1 model's tokenizer, to count context it will receive
STORE, rows_a2 = {}, []
for size in CHUNK_SIZES:
    ch = chunk_fixed(DOCS, size, tokenizer)
    search, info = build_dense_search(ch, embedder)
    hits, tok_sum, qwen_sum, split_spans = 0, [], [], 0
    for q in QUERIES:
        top = search(q["question"], FINAL_K)
        hits += any(contains(c["text"], q["span"]) for c, _ in top)
        tok_sum.append(sum(c["n_tokens"] for c, _ in top))
        qwen_sum.append(sum(len(qtok(c["text"], add_special_tokens=False)["input_ids"]) for c, _ in top))
    split_spans = sum(not any(contains(c["text"], q["span"]) for c in ch) for q in QUERIES)
    STORE[size] = {"chunks": ch, "search": search, "info": info, "hit3": 100 * hits / len(QUERIES),
                   "tpq": float(np.mean(tok_sum)), "qwen_tpq": float(np.mean(qwen_sum)), "split": split_spans}
    rows_a2.append([size, len(ch), f"{STORE[size]['hit3']:.0f}", f"{STORE[size]['tpq']:.0f}"])
md("Hit rate@3 = share of the 10 queries where one of the top 3 chunks contains the answer span. "
   "Tokens sent per query = combined size of the top 3 chunks (embedding-model tokens), averaged over the queries.\n\n" +
   md_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"], rows_a2))
for s in CHUNK_SIZES:
    print(f"{s}-token chunks: {STORE[s]['split']} of {len(QUERIES)} answer spans are cut across a chunk boundary (no chunk contains them whole) "
          f"| top-3 context = about {STORE[s]['qwen_tpq']:.0f} Assignment-1-model tokens")

In [ ]:
# ---- A2.2 choice (rule: highest hit rate@3; if equal, fewer tokens per query) ------------------------------------
CH_SIZE = max(CHUNK_SIZES, key=lambda s: (STORE[s]["hit3"], -STORE[s]["tpq"]))
others = [s for s in CHUNK_SIZES if s != CH_SIZE]; o = others[0]; S = STORE[CH_SIZE]
ft_len = 512
if (STAGE2 / "stage2_summary.json").exists():
    ft_len = json.load(open(STAGE2 / "stage2_summary.json")).get("hyperparameters", {}).get("max_seq_len", 512)
try: max_pos = AutoConfig.from_pretrained(GEN_BASE).max_position_embeddings
except Exception: max_pos = None
fits = S["qwen_tpq"] + 250 <= ft_len                                              # ~250 tokens for the prompt rules and the question
ctx = (f"This context fits within the {ft_len}-token sequence length that my Assignment 1 model was fine-tuned on" if fits else
       f"This context is longer than the {ft_len}-token sequences my Assignment 1 model was fine-tuned on" +
       (f" (its base model accepts up to {max_pos:,} tokens), so Part D checks whether it still follows the chunks" if max_pos else ""))
md(f"**Chosen chunk size for Parts B–D: {CH_SIZE} tokens.** It reached hit rate@3 of {S['hit3']:.0f}% against {STORE[o]['hit3']:.0f}% for {o}-token chunks, "
   f"and the top 3 chunks cost {S['tpq']:.0f} tokens per query (about {S['qwen_tpq']:.0f} in the Assignment 1 model's tokenizer) against {STORE[o]['tpq']:.0f} for {o}. {ctx}.")
CH, SEARCH, INFO = S["chunks"], S["search"], S["info"]
for s in others: STORE[s]["search"] = None                                       # free the unused index

In [ ]:
# ---- A1.2 report for the final chunk set -----------------------------------------------------------------------
md("**A1 report (final chunk set, " + f"{CH_SIZE}-token chunks):**\n\n" + md_table(["item", "value"], [
    ["Embedding model", EMBED_MODEL], ["Vector size", INFO["dim"]], ["Similarity measure", "cosine similarity (L2-normalised vectors, FAISS IndexFlatIP inner product)"],
    ["Batch size", INFO["batch_size"]], ["Chunks indexed", INFO["n_chunks"]], ["Index build time", f"{INFO['build_seconds']:.2f} s ({device})"]]))

## Part B — Dense vs keyword (BM25) vs hybrid (RRF)

In [ ]:
from rank_bm25 import BM25Okapi
def bm25_tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP]
bm25 = BM25Okapi([bm25_tok(c["text"]) for c in CH])

def dense_top(query, n):
    return SEARCH(query, n)
def bm25_top(query, n):
    sc = bm25.get_scores(bm25_tok(query)); idx = np.argsort(-sc, kind="stable")[:n]
    return [(CH[i], float(sc[i])) for i in idx if sc[i] > 0]                         # a chunk with no matching word is not a BM25 result
def hybrid_top(query, n):
    """Reciprocal Rank Fusion of the top-20 dense and top-20 BM25 lists: RRF(d) = 1/(60 + rank_dense) + 1/(60 + rank_bm25); ranks start at 1."""
    d = [c["id"] for c, _ in dense_top(query, FUSE_N)]; b = [c["id"] for c, _ in bm25_top(query, FUSE_N)]
    score = {}
    for lst in (d, b):
        for r, cid in enumerate(lst, 1): score[cid] = score.get(cid, 0.0) + 1.0 / (RRF_K + r)
    drank = {cid: r for r, cid in enumerate(d, 1)}
    order = sorted(score, key=lambda cid: (-score[cid], drank.get(cid, 10 ** 9), cid))[:n]
    return [(CH[cid], score[cid]) for cid in order]
METHODS = {"Dense": dense_top, "BM25": bm25_top, "Hybrid (RRF)": hybrid_top}

def first_hit_rank(res, span):
    for r, (c, _) in enumerate(res, 1):
        if contains(c["text"], span): return r
    return None

rows_b, B = [], {}
for name, fn in METHODS.items():
    fn(QUERIES[0]["question"], TOP_K)                                                  # one warm-up run, not timed
    lat, h1, h3, ranks = [], 0, 0, []
    for q in QUERIES:
        t0 = time.perf_counter(); res = fn(q["question"], TOP_K); lat.append((time.perf_counter() - t0) * 1000)
        h1 += bool(res) and contains(res[0][0]["text"], q["span"]); h3 += any(contains(c["text"], q["span"]) for c, _ in res[:3])
        ranks.append(first_hit_rank(fn(q["question"], FUSE_N), q["span"]))
    B[name] = {"lat": float(np.mean(lat)), "h1": 100 * h1 / len(QUERIES), "h3": 100 * h3 / len(QUERIES), "ranks": ranks}
    rows_b.append([name, f"{B[name]['lat']:.1f}", f"{B[name]['h1']:.0f}", f"{B[name]['h3']:.0f}"])
md(f"Top {TOP_K} chunks per query, {CH_SIZE}-token chunks, the same 10 queries. Latency = time to return the top {TOP_K} for one query, averaged over the 10 queries after one warm-up run.\n\n" +
   md_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"], rows_b))

In [ ]:
# ---- B4 best method (rule: highest hit@3, then hit@1, then lower latency) ---------------------------------------
BEST = max(B, key=lambda m: (B[m]["h3"], B[m]["h1"], -B[m]["lat"]))
big = 99
best_r = lambda i, m: B[m]["ranks"][i] or big
gaps = [(max(best_r(i, m) for m in B) - best_r(i, BEST), i) for i in range(len(QUERIES))]
g, i = max(gaps)
rk = lambda m: (str(B[m]["ranks"][i]) if B[m]["ranks"][i] else f"not in the top {FUSE_N}")
if g > 0:
    ex = (f"For query {QUERIES[i]['n']} (\"{QUERIES[i]['question']}\") the first chunk containing the answer span was at rank {rk('Dense')} for dense, "
          f"{rk('BM25')} for BM25 and {rk('Hybrid (RRF)')} for hybrid.")
else:
    ex = "All three methods ranked the answer chunk equally well for every query, so the methods differ mainly in latency."
md(f"**Best method for these queries: {BEST}** — hit rate@1 {B[BEST]['h1']:.0f}% and hit rate@3 {B[BEST]['h3']:.0f}% "
   f"(dense {B['Dense']['h1']:.0f}% / {B['Dense']['h3']:.0f}%, BM25 {B['BM25']['h1']:.0f}% / {B['BM25']['h3']:.0f}%, hybrid {B['Hybrid (RRF)']['h1']:.0f}% / {B['Hybrid (RRF)']['h3']:.0f}%), "
   f"at {B[BEST]['lat']:.1f} ms per query. {ex}")

## Part C — Reranking with a cross-encoder

In [ ]:
from sentence_transformers import CrossEncoder
ce = CrossEncoder(RERANK_MODEL, max_length=512, device=device)
fn = METHODS[BEST]
def rerank(query, cands):
    if not cands: return []
    sc = ce.predict([(query, c["text"]) for c, _ in cands], batch_size=16, show_progress_bar=False)
    return sorted(zip([c for c, _ in cands], map(float, sc)), key=lambda x: -x[1])
cands0 = fn(QUERIES[0]["question"], RERANK_N); rerank(QUERIES[0]["question"], cands0)    # warm-up

BEFORE, AFTER, rr_ms = {}, {}, []
for q in QUERIES:
    cands = fn(q["question"], RERANK_N); BEFORE[q["n"]] = cands
    t0 = time.perf_counter(); ranked = rerank(q["question"], cands); rr_ms.append((time.perf_counter() - t0) * 1000)
    AFTER[q["n"]] = ranked
mark = lambda c, q: "HIT" if contains(c["text"], q["span"]) else "miss"
for q in QUERIES:
    print(f"Query {q['n']}: {q['question']}")
    for label, lst in (("before", [c for c, _ in BEFORE[q['n']][:FINAL_K]]), ("after ", [c for c, _ in AFTER[q['n']][:FINAL_K]])):
        print(f"   {label}: " + " | ".join(f"{c['doc_id']}#{c['id']} {mark(c, q)}" for c in lst))
first_id = lambda lst: lst[0][0]["id"] if lst else None
first_hit = lambda lst, q: bool(lst) and contains(lst[0][0]["text"], q["span"])
chg = 100 * np.mean([first_id(BEFORE[q["n"]]) != first_id(AFTER[q["n"]]) for q in QUERIES])
h1_before = 100 * np.mean([first_hit(BEFORE[q["n"]], q) for q in QUERIES])
h1_after = 100 * np.mean([first_hit(AFTER[q["n"]], q) for q in QUERIES])
assert abs(h1_before - B[BEST]["h1"]) < 1e-6, "hit@1 before reranking should equal the Part B value"
md(f"Method reranked: **{BEST}** (top {RERANK_N} retrieved, rescored with `{RERANK_MODEL}`, top {FINAL_K} kept).\n\n" + md_table(["Measure", "Value"], [
    ["Reranking time per query (ms)", f"{np.mean(rr_ms):.1f}"], ["Rank change rate (%)", f"{chg:.0f}"],
    ["Hit rate@1 before reranking (from Part B)", f"{h1_before:.0f}"], ["Hit rate@1 after reranking", f"{h1_after:.0f}"]]))
verb = "raised" if h1_after > h1_before else ("did not change" if h1_after == h1_before else "lowered")
md(f"**Did reranking help?** It changed the first result for {chg:.0f}% of the queries and {verb} hit rate@1 "
   f"({h1_before:.0f}% to {h1_after:.0f}%), at a cost of {np.mean(rr_ms):.0f} ms per query on top of the {B[BEST]['lat']:.1f} ms retrieval time. " +
   ("On this small query set the extra time buys a measurable accuracy gain." if h1_after > h1_before else
    "On this small query set the extra time did not buy a measurable accuracy gain at rank 1."))

## Part D — Grounded answers and checking them
### D1. Generate grounded answers

In [ ]:
SYSTEM_GROUNDED = ("You answer questions about NIST cybersecurity guidance using ONLY the numbered chunks provided by the user.\n"
    "Rules:\n1. Use only information that is in the chunks. Do not use outside knowledge.\n"
    "2. Cite the number of each chunk you used in square brackets, for example [2].\n"
    "3. If the chunks do not contain the answer, reply exactly: Not found in the documents\n"
    "4. Answer in one to three sentences.")
def build_messages(question, chunks):
    ctx = "\n\n".join(f"[{i}] {c['text']}" for i, c in enumerate(chunks, 1))
    return [{"role": "system", "content": SYSTEM_GROUNDED}, {"role": "user", "content": f"Chunks:\n{ctx}\n\nQuestion: {question}"}]
final_chunks = lambda n: [c for c, _ in AFTER[n][:FINAL_K]]
print("=== SYSTEM PROMPT ===\n" + SYSTEM_GROUNDED)
print("\n=== USER MESSAGE TEMPLATE (shown for query 1) ===")
print(build_messages(QUERIES[0]["question"], final_chunks(1))[1]["content"][:900] + " ...")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import gc
dtype = torch.bfloat16 if device == "cuda" else torch.float32
def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
def load_generator(base, adapter=None):
    tk = AutoTokenizer.from_pretrained(base)
    if tk.pad_token is None: tk.pad_token = tk.eos_token
    m = AutoModelForCausalLM.from_pretrained(base, torch_dtype=dtype).to(device)
    if adapter is not None:
        from peft import PeftModel
        m = PeftModel.from_pretrained(m, str(adapter))
    return tk, m.eval()
@torch.no_grad()
def generate(tk, m, messages, max_new=MAX_NEW):
    prompt = tk.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tk(prompt, return_tensors="pt", add_special_tokens=False).to(m.device)
    out = m.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], max_new_tokens=max_new, do_sample=False,
                     pad_token_id=tk.pad_token_id)
    return tk.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip(), int(enc["input_ids"].shape[1])

# the one extra query whose answer is NOT in the corpus (the first candidate whose key terms are really absent is used)
CANDIDATES = [("What is the maximum fine the GDPR allows for a personal data breach?", ["gdpr"]),
              ("Which cloud provider does NIST recommend for hosting an incident response team?", ["amazon", "azure", "google cloud"]),
              ("What is the capital city of France?", ["paris"])]
alltext = " ".join(DOCS.values()).lower()
UNANS = next((q for q, terms in CANDIDATES if not any(t in alltext for t in terms)), None)
assert UNANS, "all candidate out-of-corpus questions appear in the corpus - add another"
UQ = {"n": 0, "question": UNANS}
cands_u = fn(UNANS, RERANK_N); AFTER[0] = rerank(UNANS, cands_u)                    # same pipeline: best method -> rerank -> top 3
def gen_all(tk, m):
    res = {}
    for q in QUERIES + [UQ]: res[q["n"]] = generate(tk, m, build_messages(q["question"], final_chunks(q["n"])))
    return res
def flags(res):
    cite = np.mean([bool(re.search(r"\[\d+\]", res[q["n"]][0])) for q in QUERIES])
    nf = "not found in the documents" in res[0][0].lower()
    return float(cite), nf

t0 = time.time(); tk, gm = load_generator(GEN_BASE, ADAPTER_DIR)
FT = gen_all(tk, gm); print(f"Assignment 1 model ({GEN_BASE} + adapter) generated 11 answers in {time.time() - t0:.0f} s")
cite_rate, nf_ok = flags(FT)
del gm; free_gpu()
print(f"Assignment 1 model: answers citing a chunk [n]: {cite_rate:.0%} | replies 'Not found in the documents' on the out-of-corpus query: {nf_ok}")

In [ ]:
need_switch = FALLBACK_MODE == "always" or (FALLBACK_MODE == "auto" and (cite_rate < 0.5 or not nf_ok))
GEN_USED, ANS = f"{GEN_BASE} + Assignment 1 adapter", FT
if need_switch:
    prob = next((q for q in QUERIES if not re.search(r"\[\d+\]", FT[q["n"]][0])), None)
    md("**The Assignment 1 model did not follow the grounding instructions, so I switched to an instruct model (allowed by the brief).** "
       f"Problem observed: it cited a chunk in only {cite_rate:.0%} of the 10 answers, and on the out-of-corpus query it " +
       ("replied 'Not found in the documents'." if nf_ok else "did NOT reply 'Not found in the documents'.") + "\n\nExample (query " +
       (f"{prob['n']}: {prob['question']}" if prob else "0: " + UNANS) + "):\n\n> " + esc(FT[prob["n"] if prob else 0][0]))
    if shutil.disk_usage(PROJ).free / 1e9 < 20 and not os.path.exists(FALLBACK_GEN):
        print("Note: low disk - the fallback model is deleted from the cache after use to leave room for the judge.")
    t0 = time.time(); tk, gm = load_generator(FALLBACK_GEN)
    ANS = gen_all(tk, gm); GEN_USED = FALLBACK_GEN
    del gm; free_gpu(); print(f"{FALLBACK_GEN} generated 11 answers in {time.time() - t0:.0f} s")
else:
    md("The Assignment 1 fine-tuned model followed the grounding instructions on its own, so no switch to another model was needed.")
md(f"**Generator used for the answers below: {GEN_USED}**")

In [ ]:
md("**D1 answers** (top 3 chunks after reranking, numbered [1]-[3]; greedy decoding):\n\n" +
   md_table(["#", "question", "answer", "prompt tokens"], [[q["n"], q["question"], ANS[q["n"]][0], ANS[q["n"]][1]] for q in QUERIES]))
md("**Query whose answer is not in the corpus** (checked: none of its key terms occur in the corpus text):\n\n" +
   md_table(["question", "answer", "says 'Not found in the documents'"],
            [[UNANS, ANS[0][0], "yes" if "not found in the documents" in ANS[0][0].lower() else "NO"]]))

### D2. Check the answers with an LLM judge

In [ ]:
SIZES_GB = {"Qwen/Qwen2.5-7B-Instruct": 15.5, "Qwen/Qwen2.5-3B-Instruct": 6.5}
def hub_dir(model_id): return Path(os.environ["HF_HOME"]) / "hub" / ("models--" + model_id.replace("/", "--"))
def pick_judge(exclude):
    if JUDGE_ENV:
        assert JUDGE_ENV != exclude, "the judge must differ from the generator"
        return JUDGE_ENV
    free = shutil.disk_usage(PROJ).free / 1e9
    for m in SIZES_GB:
        if m == exclude: continue
        if hub_dir(m).exists() or SIZES_GB[m] + 2 <= free: return m
    raise RuntimeError(f"Not enough free disk ({free:.1f} GB) for a judge model. Delete a1b/stage3/merged_bf16, a1b/stage2/checkpoints or unused folders in a1b/cache/hf/hub and re-run.")
gen_id = FALLBACK_GEN if need_switch else GEN_BASE
if need_switch and hub_dir(FALLBACK_GEN).exists() and shutil.disk_usage(PROJ).free / 1e9 < 20 and not JUDGE_ENV:
    shutil.rmtree(hub_dir(FALLBACK_GEN), ignore_errors=True); print("Deleted the fallback generator from the cache to free disk space.")
JUDGE = pick_judge(gen_id)
print("Judge model:", JUDGE, "| generator:", GEN_USED)
assert JUDGE != gen_id

JUDGE_SYSTEM = "You are a strict examiner. You follow the marking scheme exactly and reply only in the requested format."
JUDGE_TEMPLATE = """Check one answer from a question-answering system.

QUESTION:
{question}

EXPECTED ANSWER (the reference):
{expected}

CHUNKS the system was given:
{chunks}

SYSTEM ANSWER:
{answer}

Decide two things separately.

CORRECT - compare the system answer with the expected answer only:
- "correct": it states the key facts of the expected answer and contradicts nothing in it.
- "partly correct": it states some, but not all, of the key facts of the expected answer, or states them with a minor inaccuracy.
- "wrong": it misses the key facts, contradicts the expected answer, or is empty.
If the system answer is "Not found in the documents", it makes no claim: mark CORRECT as "wrong", because the expected answer exists, and SUPPORTED as "yes".

SUPPORTED - compare the system answer with the CHUNKS only (ignore the expected answer and ignore citation numbers such as [2]):
- "yes": every factual statement in the system answer can be found in the chunks.
- "no": at least one factual statement is not in the chunks.

Reply in exactly this format, one item per line:
CORRECT: <correct|partly correct|wrong>
SUPPORTED: <yes|no>
REASON: <one sentence; if SUPPORTED is no, quote the unsupported statement>"""
print("=== JUDGE SYSTEM PROMPT ===\n" + JUDGE_SYSTEM + "\n\n=== JUDGE PROMPT TEMPLATE ===\n" + JUDGE_TEMPLATE)

In [ ]:
def parse_judge(text):
    c = re.search(r"CORRECT\s*:\s*(partly correct|correct|wrong)", text, re.I)
    s = re.search(r"SUPPORTED\s*:\s*(yes|no)", text, re.I)
    r = re.search(r"REASON\s*:\s*(.+)", text, re.I | re.S)
    return {"correct": c.group(1).lower() if c else None, "supported": s.group(1).lower() if s else None,
            "reason": norm(r.group(1)) if r else norm(text)[:200]}

t0 = time.time()
jtk = AutoTokenizer.from_pretrained(JUDGE)
if jtk.pad_token is None: jtk.pad_token = jtk.eos_token
if device == "cuda":
    jm = AutoModelForCausalLM.from_pretrained(JUDGE, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True), device_map={"": 0}).eval()
else:
    jm = AutoModelForCausalLM.from_pretrained(JUDGE, torch_dtype=dtype).eval()
JUDGED = {}
for q in QUERIES:
    chunks_txt = "\n\n".join(f"[{i}] {c['text']}" for i, c in enumerate(final_chunks(q["n"]), 1))
    msg = [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": JUDGE_TEMPLATE.format(
        question=q["question"], expected=q["expected"], chunks=chunks_txt, answer=ANS[q["n"]][0])}]
    raw, _ = generate(jtk, jm, msg, max_new=150)                                        # greedy decoding = temperature 0
    JUDGED[q["n"]] = {**parse_judge(raw), "raw": raw}
del jm; free_gpu(); print(f"Judge ({JUDGE}) finished in {time.time() - t0:.0f} s")
unparsed = [n for n, j in JUDGED.items() if j["correct"] is None or j["supported"] is None]
if unparsed: print("WARNING: the judge reply could not be read for queries", unparsed, "- they are listed as UNPARSED and not counted")
md("**D2 judge:** " + f"`{JUDGE}` (4-bit NF4 on GPU, greedy decoding = temperature 0). Generator: `{GEN_USED}`.\n\n" +
   md_table(["#", "query", "correct", "supported", "reason"], [[q["n"], q["question"], JUDGED[q["n"]]["correct"] or "UNPARSED",
            JUDGED[q["n"]]["supported"] or "UNPARSED", JUDGED[q["n"]]["reason"]] for q in QUERIES]))
cnt = lambda key, val: sum(JUDGED[q["n"]][key] == val for q in QUERIES)
md(md_table(["Check", "Count out of 10"], [["Correct", cnt("correct", "correct")], ["Partly correct", cnt("correct", "partly correct")],
                                         ["Wrong", cnt("correct", "wrong")], ["Supported by the chunks", cnt("supported", "yes")]]))

### D2.2 Spot-check the judge (your turn)
Below are 3 answers with their chunks and the judge's verdicts. **Read them yourself**, then fill in the next cell and re-run it.

In [ ]:
SPOT = sorted(random.Random(SEED).sample([q["n"] for q in QUERIES], 3))
for n in SPOT:
    q = QUERIES[n - 1]; j = JUDGED[n]
    print("=" * 100); print(f"QUERY {n}: {q['question']}\nEXPECTED: {q['expected']}\nSYSTEM ANSWER: {ANS[n][0]}")
    for i, c in enumerate(final_chunks(n), 1): print(f"  CHUNK [{i}] {c['text']}")
    print(f"JUDGE SAYS -> correct: {j['correct']} | supported: {j['supported']} | reason: {j['reason']}")

In [ ]:
# >>> YOUR TURN: after reading the 3 items above, record your own verdict on each judge verdict ("agree" or "disagree"),
# >>> e.g. MY_VERDICTS = {3: "agree", 5: "agree", 9: "disagree"}  - the keys are the query numbers printed above.
MY_VERDICTS = {}
MY_NOTE = ""        # one sentence explaining a disagreement, if there is one
if set(MY_VERDICTS) == set(SPOT):
    agree = sum(v == "agree" for v in MY_VERDICTS.values())
    md(f"**Spot-check result:** I agree with {agree} of the judge's 3 verdicts (queries {SPOT})." + (f" Disagreement: {MY_NOTE}" if MY_NOTE else ""))
else:
    print(f"NOT FILLED IN YET - set MY_VERDICTS for queries {SPOT}, then re-run this cell.")

### D2.3 Find the cause and D2.4 suggest a fix

In [ ]:
def diagnose(q):
    """Return (kind, cause_markdown, fix_markdown) for one query, from where the answer span is in the pipeline."""
    n = q["n"]; j = JUDGED[n]; chs = final_chunks(n)
    in_chunks = [i for i, c in enumerate(chs, 1) if contains(c["text"], q["span"])]
    any_chunk = any(contains(c["text"], q["span"]) for c in CH)
    before_rank = next((r for r, (c, _) in enumerate(BEFORE[n], 1) if contains(c["text"], q["span"])), None)
    after_rank = next((r for r, (c, _) in enumerate(AFTER[n], 1) if contains(c["text"], q["span"])), None)
    if in_chunks:
        return ("generation", f"**Cause: generation.** The needed information was in the chunks given to the model (chunk [{in_chunks[0]}] contains the answer span), "
                f"yet the judge marked the answer {j['correct']} / supported={j['supported']}.",
                "**Suggested fix: change the prompt and/or the generator.** Tell the model to copy the key terms from the cited chunk and to quote the sentence it relies on, "
                "or use a larger instruct model (for example Qwen2.5-7B-Instruct). That should help because retrieval already supplied the evidence, so the error is in how the answer was written.")
    if not any_chunk:
        return ("boundary", f"**Cause: retrieval.** The answer span is not inside any single {CH_SIZE}-token chunk (fixed-size chunking cut it across a chunk boundary), "
                "so the 3 chunks could not contain the full answer.",
                "**Suggested fix: chunk with overlap or on sentence boundaries.** Overlapping chunks (or sentence-aware chunking) keep a short answer passage together in one chunk. "
                "That should help because the span is split by the fixed cut-offs.")
    if after_rank:
        return ("rank", f"**Cause: retrieval.** The needed information was among the top {RERANK_N} candidates (rank {before_rank} before reranking, rank {after_rank} after) "
                f"but outside the {FINAL_K} chunks given to the model.",
                f"**Suggested fix: send more chunks (top 5 instead of top {FINAL_K}).** The answer chunk was at rank {after_rank} after reranking, so passing 5 chunks would include it, "
                "at the cost of a longer prompt.")
    return ("depth", f"**Cause: retrieval.** None of the top {RERANK_N} chunks from {BEST} contains the answer span, although a chunk that contains it exists in the index.",
            "**Suggested fix: retrieve deeper or use a different search method.** Increasing the number of candidates before reranking, or fusing dense and BM25 "
            "results, would give the cross-encoder a chance to surface the chunk that contains the answer.")

unread = [q["n"] for q in QUERIES if JUDGED[q["n"]]["correct"] is None or JUDGED[q["n"]]["supported"] is None]
failed = [q for q in QUERIES if JUDGED[q["n"]]["correct"] == "wrong" or JUDGED[q["n"]]["supported"] == "no"]
if not failed: failed = [q for q in QUERIES if JUDGED[q["n"]]["correct"] == "partly correct"]
CASE = failed[0] if failed else None
if CASE is None:
    md("The judge found no wrong, partly correct or unsupported answer among the " + str(len(QUERIES) - len(unread)) + " it could read, so there is no failed answer to analyse." +
       (f" Its reply could not be read for queries {unread}: change the judge model or prompt and re-run from D2." if unread else ""))
else:
    n = CASE["n"]; j = JUDGED[n]
    print(f"QUERY {n}: {CASE['question']}\nEXPECTED: {CASE['expected']}\nANSWER ({GEN_USED}): {ANS[n][0]}\nJUDGE: correct={j['correct']} supported={j['supported']} | {j['reason']}\nANSWER SPAN: {CASE['span']}\n")
    for i, c in enumerate(final_chunks(n), 1): print(f"CHUNK [{i}] ({c['doc_id']}#{c['id']}): {c['text']}\n")
    kind, cause, fix = diagnose(CASE)
    md(cause + "\n\n" + fix)

## Summary

In [ ]:
summary = {"chunk_size": CH_SIZE, "embedding_model": EMBED_MODEL, "search_table": {m: {k: B[m][k] for k in ("lat", "h1", "h3")} for m in B}, "best_method": BEST,
           "rerank": {"ms": float(np.mean(rr_ms)), "rank_change_pct": float(chg), "h1_before": float(h1_before), "h1_after": float(h1_after)},
           "generator": GEN_USED, "judge": JUDGE, "judge_counts": {"correct": cnt("correct", "correct"), "partly": cnt("correct", "partly correct"),
           "wrong": cnt("correct", "wrong"), "supported": cnt("supported", "yes")}, "not_found_ok": "not found in the documents" in ANS[0][0].lower()}
(OUT / "results_2b.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
md("**Before you submit:** fill in the spot-check cell (D2.2), run all cells so every output is visible, export the HTML, then free disk space by deleting "
   "unused models under `a1b/cache/hf/hub` and stop the notebook server.")